# DeepRefine Demo（Dulce）

Pipline:
- Using **AutoSchemaKG** to construt the KB from [`data/demo/Dulce.json`](data/demo/Dulce.json).
- Given query.
- Refinement.

VLLM services needed to start:

| Model | Ports | Utils |
|------|----------|------|
| $\text{Qwen/Qwen3-4B-Instruct-2507}$ | 8132 | knowledge graph extractor |
| $\text{Qwen3-Embedding-0.6B}$ | 8128 | retriever |
| $\text{HaoyuHuang2/DeepRefine-v1-8B}$ | 8133 | refinement |

Output: `data/demo/dulce_kg_output/`(graphml、FAISS、`original_kg.pkl`、`refined_kg_*.pkl`、refine logging jsonl).


## Stage 1: Constructing the Knowledge Base.
Here we only provide the KB constructed by [AutoSchemaKG](https://github.com/HKUST-KnowComp/AutoSchemaKG).

Actually, DeepRefine can be applied in any constructed KB, in which you only need to replace the parsing interface between the actions generated by DeepRefine and the operators supported in the corresponding KB. 

In [ ]:
from __future__ import annotations

import json
import pickle
import sys
from pathlib import Path

from openai import OpenAI 

REPO_ROOT = Path("/home/haoyuhuang/www/code/DeepRefine").resolve()
DEMO_DIR = REPO_ROOT / "data" / "demo"
DULCE_JSON = DEMO_DIR / "Dulce.json"
KEYWORD = "dulce"
OUTPUT_DIR = DEMO_DIR / "dulce_kg_output"
CORPUS_JSONL = DEMO_DIR / f"{KEYWORD}.jsonl"
GRAPHML_PATH = OUTPUT_DIR / "kg_graphml" / f"{KEYWORD}_graph.graphml"
ORIGINAL_KG_PKL = OUTPUT_DIR / "original_kg.pkl"

for p in (REPO_ROOT / "AutoSchemaKG", REPO_ROOT):
    ps = str(p)
    if ps not in sys.path:
        sys.path.insert(0, ps)

from atlas_rag.kg_construction.triple_config import ProcessingConfig
from atlas_rag.kg_construction.triple_extraction import KnowledgeGraphExtractor
from atlas_rag.llm_generator import GenerationConfig, LLMGenerator
from atlas_rag.vectorstore.create_graph_index import create_embeddings_and_index
from atlas_rag.vectorstore.embedding_model import Qwen3Emb

CFG = {
    "kg_llm_base_url": "http://0.0.0.0:8132/v1",
    "embed_base_url": "http://0.0.0.0:8128/v1",
    "kg_model": "Qwen/Qwen3-4B-Instruct-2507",
    "encoder_model": "Qwen/Qwen3-Embedding-0.6B",
}
PROMPT_DIR = REPO_ROOT / "benchmark" / "autograph"

# Dulce.json 为 JSON 数组；抽取管线使用 jsonl（每行一条 passage）
if not CORPUS_JSONL.exists() or CORPUS_JSONL.stat().st_mtime < DULCE_JSON.stat().st_mtime:
    with DULCE_JSON.open(encoding="utf-8") as f:
        passages = json.load(f)
    with CORPUS_JSONL.open("w", encoding="utf-8") as f:
        for row in passages:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
    print(f"Wrote corpus: {CORPUS_JSONL} ({len(passages)} passages)")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not GRAPHML_PATH.exists():
    print("Running AutoSchemaKG extraction (LLM triples → csv → graphml)...")
    kg_client = OpenAI(base_url=CFG["kg_llm_base_url"], api_key="EMPTY")
    kg_llm = LLMGenerator(
        client=kg_client,
        model_name=CFG["kg_model"],
        max_workers=4,
        default_config=GenerationConfig(temperature=0.7),
    )
    kg_config = ProcessingConfig(
        model_path=CFG["kg_model"],
        data_directory=str(DEMO_DIR),
        filename_pattern=KEYWORD,
        batch_size_triple=8,
        batch_size_concept=8,
        output_directory=str(OUTPUT_DIR),
        max_new_tokens=8192,
        max_workers=4,
        remove_doc_spaces=True,
        include_concept=False,
        triple_extraction_prompt_path=str(PROMPT_DIR / "custom_prompt.json"),
        triple_extraction_schema_path=str(PROMPT_DIR / "custom_schema.json"),
        record=False,
    )
    extractor = KnowledgeGraphExtractor(model=kg_llm, config=kg_config)
    extractor.run_extraction()
    extractor.convert_json_to_csv()
    extractor.convert_to_graphml()
    print(f"GraphML: {GRAPHML_PATH}")
else:
    print(f"Skip extraction, graph exists: {GRAPHML_PATH}")

if not ORIGINAL_KG_PKL.exists() or ORIGINAL_KG_PKL.stat().st_mtime < GRAPHML_PATH.stat().st_mtime:
    embed_client = OpenAI(base_url=CFG["embed_base_url"], api_key="EMPTY")
    sentence_encoder = Qwen3Emb(embed_client)
    demo_data = create_embeddings_and_index(
        sentence_encoder=sentence_encoder,
        model_name=CFG["encoder_model"],
        working_directory=str(OUTPUT_DIR),
        keyword=KEYWORD,
        include_concept=False,
        include_events=False,
        normalize_embeddings=False,
        text_batch_size=64,
        node_and_edge_batch_size=64,
        use_flat_index=True,
    )
    with ORIGINAL_KG_PKL.open("wb") as f:
        pickle.dump(demo_data, f)
    print(f"Saved: {ORIGINAL_KG_PKL}")
else:
    with ORIGINAL_KG_PKL.open("rb") as f:
        demo_data = pickle.load(f)
    print(f"Loaded cached KG bundle: {ORIGINAL_KG_PKL}")

print(f"KG nodes={demo_data['KG'].number_of_nodes()}, edges={demo_data['KG'].number_of_edges()}")


/home/haoyuhuang/miniconda3/envs/atlastune/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/haoyuhuang/miniconda3/envs/atlastune/lib/python3.10/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
/home/haoyuhuang/miniconda3/envs/atlastune/lib/python3.10/site-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


Wrote corpus: /home/haoyuhuang/www/code/DeepRefine/data/demo/dulce.jsonl (3 passages)
Running AutoSchemaKG extraction (LLM triples → csv → graphml)...
Initialized LLMGenerator with inference_type='api', backend='openai'
Using custom kg extraction prompt:
{'en': {'system': 'You are a helpful assistant', 'triple_extraction': 'You are an expert knowledge graph constructor.\nYour task is to extract factual information from the provided text and represent it strictly as a ***JSON array*** of knowledge graph triples.\n\n### Output Format\n- The output must be a **JSON array**.\n- Each element in the array must be a **JSON object** with exactly three non-empty keys:\n  - "subject": the main entity, concept, event, or attribute.\n  - "relation": a concise, descriptive phrase or verb that describes the relationship (e.g., "founded by", "started on", "is a", "has circulation of").\n  - "object": the entity, concept, value, event, or attribute that the subject has a relationship with.\n\n### Cons

Generating train split: 3 examples [00:00, 781.35 examples/s]


Processing shard 1/1 (texts 0-2 of 3, 3 documents)
Generated 3 chunks for shard 1/1
Model: Qwen/Qwen3-4B-Instruct-2507


  0%|          | 0/1 [00:00<?, ?it/s]

...

Computing text embeddings...


100%|██████████| 1/1 [00:00<00:00, 7194.35it/s]


Node and edge embeddings not found, computing...


100%|██████████| 3/3 [00:00<00:00, 3321.78it/s]

Node and edge embeddings already computed.
Saved: /home/haoyuhuang/www/code/DeepRefine/data/demo/dulce_kg_output/original_kg.pkl
KG nodes=79, edges=153


## Online Queries
Assume there are some random online queries.

In [6]:
QUERIES = [
    {
        "id": "mh_q1",
        "question": "Who analyzed Dulce transmission logs, repelled a drone at the hidden panel, and later called the signals a structured anomaly?",
    },
    {
        "id": "mh_q2",
        "question": "Who warned Alex the team were pawns, flagged a recent panel cover-up, and studied crash-site alien tech that could change physics?",
    },
    {
        "id": "mh_q3",
        "question": "Who rebuked speculation in the briefing, ordered elevator systems checks underground, then showed reverence for the alien device in the lab?",
    },
]

for q in QUERIES:
    print(q["id"], ":", q["question"])


mh_q1 : Who analyzed Dulce transmission logs, repelled a drone at the hidden panel, and later called the signals a structured anomaly?
mh_q2 : Who warned Alex the team were pawns, flagged a recent panel cover-up, and studied crash-site alien tech that could change physics?
mh_q3 : Who rebuked speculation in the briefing, ordered elevator systems checks underground, then showed reverence for the alien device in the lab?


## Stage 2: Refinement

In [10]:
import time
from autorefiner.src.reafiner import Reafiner, RetrievalStepResult

CFG_REFINE = {
    "reafiner_base_url": "http://0.0.0.0:8134/v1",
    "reafiner_model": "HaoyuHuang2/DeepRefine-v1-8B",
}
REFINED_PKL = OUTPUT_DIR / f"refined_kg_{CFG_REFINE['reafiner_model'].replace('/', '_')}.pkl"
REFINE_LOG = OUTPUT_DIR / f"refinement_results_{int(time.time())}.jsonl"


def _refinement_result_to_jsonable(sample: dict, final_answer, refinement_result) -> dict:
    base = {"sample": sample, "final_answer": final_answer}
    if refinement_result is None:
        base["refinement_result"] = None
        return base

    hist = []
    for step in refinement_result.interaction_history:
        if isinstance(step, RetrievalStepResult):
            hist.append(
                {
                    "num_hops": step.num_hops,
                    "base_top_k": step.base_top_k,
                    "query": step.query,
                    "retrieved_subgraph": step.retrieved_subgraph,
                    "raw_response": step.raw_response,
                    "answerable": step.answerable,
                    "answer": step.answer,
                }
            )
        else:
            hist.append(str(step))

    base["refinement_result"] = {
        "query": refinement_result.query,
        "history_horizon_size": refinement_result.history_horizon_size,
        "interaction_history": hist,
        "error_abduction_reason": refinement_result.error_abduction_reason,
        "original_subgraph": refinement_result.original_subgraph,
        "refined_subgraph": refinement_result.refined_subgraph,
        "refinement_action_raw": refinement_result.refinement_action_raw,
        "refinement_action_count": len(refinement_result.refinement_action_list),
    }
    return base

# load demo data
if "demo_data" not in globals():
    with ORIGINAL_KG_PKL.open("rb") as f:
        demo_data = pickle.load(f)

# vllm service
reafiner_client = OpenAI(base_url=CFG_REFINE["reafiner_base_url"], api_key="EMPTY")
reafiner_llm = LLMGenerator(
    client=reafiner_client,
    model_name=CFG_REFINE["reafiner_model"],
    default_config=GenerationConfig(chat_template_kwargs={"enable_thinking": False}),
)

if "sentence_encoder" not in globals():
    embed_client = OpenAI(base_url=CFG["embed_base_url"], api_key="EMPTY")
    sentence_encoder = Qwen3Emb(embed_client)

# deeprefine
reafiner = Reafiner(
    data=demo_data,
    sentence_encoder=sentence_encoder,
    llm_generator=reafiner_llm,
    base_top_k=5,
    max_hops=4,
    max_triple_num=20,
    max_triple_num_by_step=[5, 10, 15, 20],
    history_horizon_size=4,
    if_gen_answer=False,
)

print(f"Refining {len(QUERIES)} queries → {REFINE_LOG}")
with REFINE_LOG.open("w", encoding="utf-8") as log_f:
    for sample in QUERIES:
        query = sample["question"]
        print("\n" + "=" * 60)
        print(f"[{sample['id']}] {query}")
        final_answer, _, refinement_result = reafiner.refine(query=query)
        record = _refinement_result_to_jsonable(sample, final_answer, refinement_result)
        log_f.write(json.dumps(record, ensure_ascii=False) + "\n")
        log_f.flush()
        n_steps = (
            len(refinement_result.interaction_history)
            if refinement_result is not None
            else 0
        )
        print(f"  steps={n_steps}, nodes={reafiner.kg.number_of_nodes()}, edges={reafiner.kg.number_of_edges()}")

# write back passage nodes (specific for AutoSchemaKG)
for text_id in list(reafiner.text_id_to_node_name.keys()):
    reafiner.kg.add_node(
        text_id,
        file_id=text_id,
        id=reafiner._safe_sanitize(reafiner.text_id_to_node_name[text_id]),
        type="passage",
    )
for node_id in list(reafiner.node_list):
    if reafiner.node_id_to_file_id[node_id] is not None:
        reafiner.kg.add_edge(
            node_id,
            reafiner.node_id_to_file_id[node_id],
            relation="mention in",
            type="Source",
        )

demo_data = reafiner.data
demo_data["KG"] = reafiner.kg
with REFINED_PKL.open("wb") as f:
    pickle.dump(demo_data, f)

print(f"\nDone. Log: {REFINE_LOG}\nRefined KG: {REFINED_PKL}")


Initialized LLMGenerator with inference_type='api', backend='openai'
Refining 3 queries → /home/haoyuhuang/www/code/DeepRefine/data/demo/dulce_kg_output/refinement_results_1780314376.jsonl

[mh_q1] Who analyzed Dulce transmission logs, repelled a drone at the hidden panel, and later called the signals a structured anomaly?
 [Step: 1] 
<judge>No</judge>
 [Step: 2] 
<judge>Yes</judge>
<refinement>insert_edge("Sam Rivera", "analyzed", "Dulce transmission logs")|insert_edge("Sam Rivera", "repelled", "drone at hidden panel")|insert_edge("Sam Rivera", "called", "signals a structured anomaly")</refinement>
  steps=2, nodes=82, edges=84

[mh_q2] Who warned Alex the team were pawns, flagged a recent panel cover-up, and studied crash-site alien tech that could change physics?
 [Step: 1] 
<judge>Yes</judge>
  steps=1, nodes=82, edges=84

[mh_q3] Who rebuked speculation in the briefing, ordered elevator systems checks underground, then showed reverence for the alien device in the lab?
 [Step: 1] 
